Pour isoler deux tickets de support différents dans votre application LangChain, la méthode standard consiste à utiliser un identifiant unique appelé thread_id. Cet identifiant permet au système de suivi (le checkpointer) de distinguer et de stocker séparément l'historique de chaque conversation.
Voici comment mettre en œuvre cette isolation en suivant les principes de vos sources :
## 1. Le rôle du thread_id
Le thread_id agit comme une clé d'accès à une session spécifique dans la mémoire de l'agent.
- Isolation complète : Si vous utilisez un thread_id différent pour chaque ticket (par exemple TICKET-123 et TICKET-456), l'agent traitera chaque demande comme une nouvelle conversation indépendante
- Persistance : Les échanges liés à un ID sont conservés par un composant de sauvegarde (comme InMemorySaver), permettant de reprendre une discussion là où elle s'était arrêtée pour un ticket précis

In [ ]:
import os
from langchain_ollama import ChatOllama 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
llm = ChatOllama(model="gemma3:4b", base_url=OLLAMA_HOST, temperature=0) # Ou tout modèle supportant cette fonction
# Le prompt est un "moule" avec des trous {variable} qu'on remplira plus tard
prompt = ChatPromptTemplate.from_template(
    "Explique le concept de {concept} en une seule phrase claire et pédagogique."
)

# Le parser convertit la réponse du LLM (objet AIMessage complexe) en simple string
parser = StrOutputParser()

# 3.  on assemble avec | (lire de gauche à droite)
#    prompt -> llm -> parser
chaine = prompt | llm | parser

In [ ]:
# Configuration pour le Ticket A
config_ticket_A = {"configurable": {"thread_id": "TICKET-123"}}

# Configuration pour le Ticket B
config_ticket_B = {"configurable": {"thread_id": "TICKET-456"}}

In [ ]:
# Appel pour le Ticket A : l'IA mémorise ce contexte
chaine.invoke({"messages": "Le wifi est coupé à Bruxelles"}, config=config_ticket_A)

# Appel pour le Ticket B : l'IA repart de zéro, sans savoir pour Bruxelles (Isolation)
chaine.invoke({"messages": "J'ai besoin d'un nouveau clavier"}, config=config_ticket_B)